# Notebook 10: Infinity-2B GGUF Colab smoke tests

This notebook loads the unofficial Q8_0 GGUF port of Infinity-2B and runs inference-only prompt tests. It is designed for a free Colab or a small GPU workspace.

The GGUF port is a separate loader from the official PyTorch checkpoint. It uses the quantized Infinity model and T5 encoder GGUF files, while the VAE and the Infinity Python architecture come from the official repository. This notebook does not implement PFB, SAC, or style injection yet.

In [ ]:
from pathlib import Path
import gc
import importlib.util
import os
import re
import shutil
import subprocess
import sys
import time

# All files are kept under one directory so the notebook can be rerun safely.
ROOT = Path('/content/notebook_10_infinity_2b_gguf')
PORT_DIR = ROOT / 'gguf_port'
OFFICIAL_DIR = PORT_DIR / 'Infinity'
ASSET_DIR = ROOT / 'assets'
OUTPUT_DIR = ROOT / 'outputs'
for path in (PORT_DIR, ASSET_DIR, OUTPUT_DIR):
    path.mkdir(parents=True, exist_ok=True)

OFFICIAL_REPO = 'https://github.com/FoundationVision/Infinity.git'
GGUF_REPO = 'kzopp/Infinity-2B-GGUF_UNOFFICIAL'
MODEL_PN = '0.25M'  # 0.06M ~= 256px, 0.25M ~= 512px, 1M ~= 1024px
CFG_SCALE = 3.0
TAU = 0.5
SEED = 42
print('ROOT:', ROOT)
print('MODEL_PN:', MODEL_PN, '| CFG:', CFG_SCALE, '| TAU:', TAU, '| SEED:', SEED)

In [ ]:
# Check the runtime before installing anything.
import torch

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print('GPU:', props.name)
    print('VRAM GiB:', round(props.total_memory / 2**30, 2))
else:
    print('WARNING: no GPU detected; inference will be extremely slow.')

In [ ]:
# Install the packages used by the GGUF loader.
# We intentionally do not install torch or flash-attn here: Colab already ships torch,
# and the GGUF loader falls back to PyTorch SDPA when flash-attn is unavailable.
packages = [
    'gguf', 'gradio', 'transformers', 'sentencepiece',
    'easydict', 'typed-argument-parser', 'seaborn', 'kornia',
    'gputil', 'colorama', 'omegaconf', 'timm==0.9.6',
    'decord', 'pytz', 'imageio', 'einops', 'opencv-python',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)
print('Dependency installation finished.')

In [ ]:
# Clone the official Python architecture only once.
if not OFFICIAL_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', OFFICIAL_REPO, str(OFFICIAL_DIR)], check=True)
else:
    print('Official Infinity source already exists:', OFFICIAL_DIR)

# Download only the files needed from the unofficial GGUF repository.
from huggingface_hub import hf_hub_download

def download_hf_file(filename, target_dir):
    target_dir.mkdir(parents=True, exist_ok=True)
    return Path(hf_hub_download(
        repo_id=GGUF_REPO,
        filename=filename,
        local_dir=str(target_dir),
    ))

PORT_SCRIPT = download_hf_file('generate_image_2b_q8_gguf.py', PORT_DIR)
PORT_UTILS = download_hf_file('infinity_gguf_utils.py', PORT_DIR)
PATCH_DIR = ROOT / 'gguf_patched_source'
PATCHED_BASIC = download_hf_file('Infinity/infinity/models/basic.py', PATCH_DIR)
PATCHED_INFINITY = download_hf_file('Infinity/infinity/models/infinity.py', PATCH_DIR)

# The GGUF repository includes patched source files with optional attention fallbacks.
# Copy them over the matching files in the official source tree.
official_basic = OFFICIAL_DIR / 'infinity' / 'models' / 'basic.py'
official_infinity = OFFICIAL_DIR / 'infinity' / 'models' / 'infinity.py'
shutil.copy2(PATCHED_BASIC, official_basic)
shutil.copy2(PATCHED_INFINITY, official_infinity)
INFINITY_GGUF = download_hf_file('infinity_2b_reg_Q8_0.gguf', ASSET_DIR)
T5_GGUF = download_hf_file('flan-t5-xl-encoder-Q8_0.gguf', ASSET_DIR)
VAE_PATH = download_hf_file('Infinity/infinity_vae_d32_reg.pth', ASSET_DIR)

print('GGUF model:', INFINITY_GGUF)
print('T5 encoder:', T5_GGUF)
print('VAE:', VAE_PATH)
print('Loader:', PORT_SCRIPT)
print('Loader utility:', PORT_UTILS)

In [ ]:
# Verify the expected files before importing the custom loader.
required_files = [PORT_SCRIPT, PORT_UTILS, PATCHED_BASIC, PATCHED_INFINITY, INFINITY_GGUF, T5_GGUF, VAE_PATH]
missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required files:\n' + '\n'.join(missing))

for path in required_files:
    print(f'{path.name:40s} {path.stat().st_size / 2**30:.3f} GiB')

assert (OFFICIAL_DIR / 'infinity' / 'models' / 'infinity.py').exists(), 'Official Infinity source is incomplete.'
print('All GGUF, VAE, and official source files are present.')

## Import the unofficial loader

The upstream GGUF script contains a NumPy 2 compatibility assignment to `np.ndarray.newbyteorder`. That assignment can fail on some Colab runtimes because NumPy types are immutable. The next cell creates a temporary sanitized copy of the loader and removes only that obsolete compatibility block.

In [ ]:
sys.path.insert(0, str(PORT_DIR))
sys.path.insert(0, str(OFFICIAL_DIR))

loader_source = PORT_SCRIPT.read_text()
compat_pattern = r"\n    # Apply NumPy 2\.0 compatibility patch.*?\n    # Load GGUF state dict"
loader_source, replacements = re.subn(
    compat_pattern,
    '\n    # NumPy compatibility is handled by the installed gguf package.\n    # Load GGUF state dict',
    loader_source,
    count=1,
    flags=re.S,
)
print('Removed obsolete NumPy compatibility block:', replacements == 1)

PATCHED_LOADER = PORT_DIR / 'generate_image_2b_q8_gguf_colab.py'
PATCHED_LOADER.write_text(loader_source)
spec = importlib.util.spec_from_file_location('infinity_gguf_colab_loader', PATCHED_LOADER)
gguf_loader = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = gguf_loader
spec.loader.exec_module(gguf_loader)
print('Custom GGUF loader imported successfully.')

## Load all components

The T5 encoder stays on CPU to reduce VRAM usage. The VAE and quantized Infinity transformer are placed on the GPU.

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE != 'cuda':
    raise RuntimeError('A CUDA GPU is required for practical inference. Select a GPU runtime and rerun.')

print('[1/4] Loading VAE...')
vae = gguf_loader.load_vae(str(VAE_PATH), vae_type=32, device=DEVICE)

print('[2/4] Loading T5 tokenizer...')
text_tokenizer = gguf_loader.load_t5_tokenizer_from_gguf(str(T5_GGUF))

print('[3/4] Loading quantized T5 encoder on CPU...')
text_encoder = gguf_loader.load_t5_encoder_from_gguf(str(T5_GGUF), device='cpu')

print('[4/4] Loading quantized Infinity-2B transformer on GPU...')
infinity_model = gguf_loader.load_infinity_from_gguf(
    str(INFINITY_GGUF),
    vae=vae,
    device=DEVICE,
    model_type='infinity_2b',
    text_channels=2048,
    pn=MODEL_PN,
)

infinity_model.eval()
vae.eval()
print('All components loaded successfully.')

In [ ]:
# Build the official dynamic-resolution schedule for the selected preset.
import numpy as np
from infinity.utils.dynamic_resolution import dynamic_resolution_h_w, h_div_w_templates

ASPECT_RATIO = 1.0
h_div_w_template = h_div_w_templates[np.argmin(np.abs(h_div_w_templates - ASPECT_RATIO))]
scale_schedule = dynamic_resolution_h_w[h_div_w_template][MODEL_PN]['scales']
scale_schedule = [(1, h, w) for (_, h, w) in scale_schedule]
print('Aspect ratio:', h_div_w_template)
print('Preset:', MODEL_PN)
print('Scale schedule:', scale_schedule)

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

def tensor_to_pil(image):
    """Convert common Infinity output layouts/ranges into an RGB PIL image."""
    if isinstance(image, (list, tuple)):
        image = image[0]
    tensor = image.detach().float().cpu() if torch.is_tensor(image) else torch.as_tensor(image).float()
    if tensor.ndim == 4:
        tensor = tensor[0]
    if tensor.ndim != 3:
        raise ValueError(f'Unexpected image shape: {tuple(tensor.shape)}')
    if tensor.shape[0] in (1, 3, 4):
        tensor = tensor.permute(1, 2, 0)
    if tensor.shape[-1] == 1:
        tensor = tensor.repeat(1, 1, 3)
    if tensor.shape[-1] > 3:
        tensor = tensor[..., :3]
    lo, hi = float(tensor.min()), float(tensor.max())
    if lo < -0.05:
        tensor = (tensor + 1.0) / 2.0
    elif hi > 1.05:
        tensor = tensor / 255.0
    array = (tensor.clamp(0, 1).numpy() * 255).round().astype('uint8')
    return Image.fromarray(array, mode='RGB')

def generate_one(prompt, seed=SEED, output_path=None):
    started = time.time()
    with torch.inference_mode():
        image = gguf_loader.generate_image(
            infinity_model, vae, text_tokenizer, text_encoder, prompt,
            cfg_scale=CFG_SCALE,
            tau=TAU,
            seed=seed,
            scale_schedule=scale_schedule,
            vae_type=32,
            device=DEVICE,
        )
    pil = tensor_to_pil(image)
    if output_path is not None:
        pil.save(output_path)
    del image
    gc.collect()
    torch.cuda.empty_cache()
    print(f'Generated in {time.time() - started:.2f}s:', prompt)
    return pil

In [ ]:
# Single-image smoke test.
SMOKE_PROMPT = 'a ceramic teapot on a wooden table'
smoke_path = OUTPUT_DIR / 'smoke_test.png'
smoke_image = generate_one(SMOKE_PROMPT, seed=SEED, output_path=smoke_path)
display(smoke_image)
print('Saved:', smoke_path)

## Prompt tests

These prompts contain only objects and scenes so the first run focuses on basic content generation. Change `MODEL_PN` above to `1M` only after the 512px smoke test works.

In [ ]:
TEST_PROMPTS = [
    'a red apple on a wooden table',
    'a lighthouse beside the sea',
    'a steam train traveling through green countryside',
    'a blue bicycle leaning against a brick wall',
    'a mountain lake beneath a bright sky',
    'a colorful hot air balloon above a valley',
    'a stone castle reflected in a lake',
    'a vintage camera on a desk',
]

def safe_name(text):
    return re.sub(r'[^a-z0-9]+', '_', text.lower()).strip('_')[:70]

test_results = []
for index, prompt in enumerate(TEST_PROMPTS):
    output_path = OUTPUT_DIR / f'{index:02d}_{safe_name(prompt)}_seed{SEED}.png'
    pil = generate_one(prompt, seed=SEED, output_path=output_path)
    test_results.append({'prompt': prompt, 'path': output_path, 'image': pil})

print(f'Generated {len(test_results)} images under {OUTPUT_DIR}')

In [ ]:
# Contact sheet for visual inspection.
ncols = 4
nrows = int(np.ceil(len(test_results) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4 * nrows))
axes = np.atleast_1d(axes).ravel()
for ax, result in zip(axes, test_results):
    ax.imshow(result['image'])
    ax.set_title(result['prompt'], fontsize=10)
    ax.axis('off')
for ax in axes[len(test_results):]:
    ax.axis('off')
fig.suptitle(f'Infinity-2B GGUF prompt tests | {MODEL_PN} | CFG {CFG_SCALE} | tau {TAU} | seed {SEED}', fontsize=14)
fig.tight_layout()
contact_path = OUTPUT_DIR / f'prompt_tests_{MODEL_PN}_seed{SEED}.png'
fig.savefig(contact_path, dpi=150, bbox_inches='tight')
plt.show()
print('Saved contact sheet:', contact_path)

## Troubleshooting

- `CUDA out of memory`: set `MODEL_PN = '0.06M'`, keep one prompt at a time, restart the runtime, and do not install SageAttention until the basic path works.
- `flash_attn` or `sageattention` installation fails: skip it. The loader falls back to PyTorch SDPA.
- Missing VAE: verify that `VAE_PATH` points to `assets/Infinity/infinity_vae_d32_reg.pth`.
- Missing official modules: rerun the clone cell and check that `gguf_port/Infinity/infinity/models/bsq_vae` exists.
- Slow first run: the tokenizer/config and model files are cached after the first download.
- This port hardcodes `top_k=900` and `top_p=0.97` inside its generation function. The current notebook exposes CFG, tau, seed, aspect ratio, and `pn`; it does not claim support for the official PyTorch sampling controls.